# LLM 급식 메뉴 추천 RAG 프로젝트

## 1. 데이터 로드

급식 메뉴, 메뉴별 식재료, 식품 영양성분 데이터를 결합하여
LLM/RAG 검색에 사용할 메뉴 데이터셋을 구축한다.

In [ ]:
import re
import numpy as np
import pandas as pd

from pathlib import Path

In [ ]:
# 현재 프로젝트 위치
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"

print("현재 작업 폴더 :", BASE_DIR)
print("dataset 경로  :", DATA_DIR)
print("dataset 존재  :", DATA_DIR.exists())

print("\n=== dataset 파일 목록 ===")

for file in DATA_DIR.iterdir():
    print(f"{file.name:40} {file.stat().st_size:,} bytes")

현재 작업 폴더 : d:\pythonscript\LLM-PRJ
dataset 경로  : d:\pythonscript\LLM-PRJ\dataset
dataset 존재  : True

=== dataset 파일 목록 ===
food_nutrition.xlsx                      13,348,408 bytes
menugen_menu_ingredients.csv             3,361,795 bytes
menugen_menu_master.csv                  218,450 bytes


## 2. 메뉴 마스터 데이터 확인

MenuGen 메뉴 마스터 데이터를 불러오고
메뉴 수, 컬럼 구조, 결측치 여부를 확인한다.

In [3]:
menu_path = DATA_DIR / "menugen_menu_master.csv"

menu_df = pd.read_csv(menu_path)

print("menu_df shape :", menu_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(menu_df.columns):
    print(i, col)

print("\n=== 상위 5행 ===")
display(menu_df.head())

print("\n=== 결측치 개수 ===")
display(menu_df.isnull().sum())

menu_df shape : (3250, 7)

=== 컬럼 목록 ===
0 no
1 fd_Code
2 upper_Fd_Grupp_Nm
3 fd_Grupp_Nm
4 fd_Nm
5 fd_Wgh
6 food_Cnt

=== 상위 5행 ===


,no,fd_Code,upper_Fd_Grupp_Nm,fd_Grupp_Nm,fd_Nm,fd_Wgh,food_Cnt
0,1,D011001,밥류,쌀밥,눌은밥,440.0,1
1,2,D011002,밥류,쌀밥,쌀밥,210.0,1
2,3,D011003,밥류,쌀밥,찰밥,210.0,1
3,4,D011004,밥류,쌀밥,현미밥,210.0,1
4,5,D011006,밥류,쌀밥,현미밥,160.0,1



=== 결측치 개수 ===


no                   0
fd_Code              0
upper_Fd_Grupp_Nm    0
fd_Grupp_Nm          0
fd_Nm                0
fd_Wgh               0
food_Cnt             0
dtype: int64

## 3. 메뉴별 식재료 데이터 확인

메뉴 코드(`fd_Code`)를 기준으로 각 메뉴에 연결된 식재료 데이터를 불러오고,
행 수, 컬럼 구조, 결측치를 확인한다.

In [5]:
ingredient_path = DATA_DIR / "menugen_menu_ingredients.csv"

ingredient_df = pd.read_csv(ingredient_path)

print("ingredient_df shape :", ingredient_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(ingredient_df.columns):
    print(i, col)

print("\n=== 상위 10행 ===")
display(ingredient_df.head(10))

print("\n=== 결측치 개수 ===")
display(ingredient_df.isnull().sum())

print("\n=== 고유 메뉴 코드 수 ===")
print(ingredient_df["menu_fd_Code"].nunique())

ingredient_df shape : (22582, 15)

=== 컬럼 목록 ===
0 menu_fd_Code
1 menu_fd_Nm
2 menu_upper_Fd_Group_Nm
3 menu_fd_Group_Nm
4 menu_fd_Wgh
5 ingredient_fd_Code
6 ingredient_food_Code
7 ingredient_food_Nm
8 ingredient_fd_Eng_Nm
9 ingredient_nation_Std_Food_Grupp_Code_Nm
10 ingredient_origin_Code_Nm
11 ingredient_food_Wgh
12 ingredient_allrgy_Info
13 ingredient_onslf_Std_Food_Grupp_Nm
14 ingredient_amplt_Cl_Nm

=== 상위 10행 ===


,menu_fd_Code,menu_fd_Nm,menu_upper_Fd_Group_Nm,menu_fd_Group_Nm,menu_fd_Wgh,ingredient_fd_Code,ingredient_food_Code,ingredient_food_Nm,ingredient_fd_Eng_Nm,ingredient_nation_Std_Food_Grupp_Code_Nm,ingredient_origin_Code_Nm,ingredient_food_Wgh,ingredient_allrgy_Info,ingredient_onslf_Std_Food_Grupp_Nm,ingredient_amplt_Cl_Nm
0,D011001,눌은밥,NaN,NaN,440.0,D011001,F00093,"즉석밥, 누룽지, 끓는물 부음","Instant cooked rice, Scorched rice(Nurungji), ...",곡류 및 그 제품,농진청,440.0,NaN,NaN,NaN
1,D011002,쌀밥,NaN,NaN,210.0,D011002,F00079,"멥쌀, 백미, 밥","Rice(Ssal), White, Cooked",곡류 및 그 제품,농진청,210.0,NaN,NaN,NaN
2,D011003,찰밥,NaN,NaN,210.0,D011003,F03123,"찹쌀, 백미, 보람찰, 밥","Rice(Ssal), Glutinous, White, Boramchal, Cooked",곡류 및 그 제품,농진청,210.0,NaN,NaN,NaN
3,D011004,현미밥,NaN,NaN,210.0,D011004,F00081,"멥쌀, 현미, 밥","Rice(Ssal), Brown, Cooked",곡류 및 그 제품,농진청,210.0,NaN,NaN,NaN
4,D011006,현미밥,NaN,NaN,160.0,D011006,F00081,"멥쌀, 현미, 밥","Rice(Ssal), Brown, Cooked",곡류 및 그 제품,농진청,160.0,NaN,NaN,NaN
5,D011007,누룽지(멥쌀),NaN,NaN,100.0,D011007,F00014,메밀묵,"Memilmuk, Buckwheat starch jelly",곡류 및 그 제품,농진청,100.0,메밀,NaN,NaN
6,D011008,흑미밥,NaN,NaN,90.0,D011008,F00017,"멥쌀, 백미, 생것","Rice(Ssal), White, Raw",곡류 및 그 제품,대푯값,70.0,NaN,NaN,NaN
7,D011008,흑미밥,NaN,NaN,90.0,D011008,F03154,"멥쌀, 현미, 흑미, 생것","Rice(Ssal), Black, Raw",곡류 및 그 제품,농진청,20.0,NaN,NaN,NaN
8,D011009,발아현미밥,NaN,NaN,100.0,D011009,F00017,"멥쌀, 백미, 생것","Rice(Ssal), White, Raw",곡류 및 그 제품,대푯값,90.0,NaN,NaN,NaN
9,D011009,발아현미밥,NaN,NaN,100.0,D011009,F00025,"멥쌀, 현미, 발아현미, 생것","Rice(Ssal), Brown, Germinated, Raw",곡류 및 그 제품,농진청,10.0,NaN,NaN,NaN



=== 결측치 개수 ===


menu_fd_Code                                    0
menu_fd_Nm                                      0
menu_upper_Fd_Group_Nm                      22582
menu_fd_Group_Nm                            22582
menu_fd_Wgh                                     0
ingredient_fd_Code                              0
ingredient_food_Code                            0
ingredient_food_Nm                              0
ingredient_fd_Eng_Nm                            0
ingredient_nation_Std_Food_Grupp_Code_Nm        0
ingredient_origin_Code_Nm                    1429
ingredient_food_Wgh                             0
ingredient_allrgy_Info                      17036
ingredient_onslf_Std_Food_Grupp_Nm          22582
ingredient_amplt_Cl_Nm                      22582
dtype: int64


=== 고유 메뉴 코드 수 ===
3250


## 4. 메뉴 마스터와 식재료 데이터 연결 검증

메뉴 코드 기준으로 두 데이터셋이 정상적으로 대응되는지 확인하고,
식재료가 없는 메뉴 또는 메뉴 마스터에 존재하지 않는 코드가 있는지 검증한다.

In [7]:
# 메뉴 마스터 코드
menu_codes = set(menu_df["fd_Code"])

# 식재료 데이터의 메뉴 코드
ingredient_menu_codes = set(ingredient_df["menu_fd_Code"])

print("메뉴 마스터 고유 코드 수 :", len(menu_codes))
print("식재료 데이터 고유 코드 수 :", len(ingredient_menu_codes))

# 메뉴 마스터에는 있지만 식재료 데이터에는 없는 메뉴
missing_ingredient_codes = menu_codes - ingredient_menu_codes

# 식재료 데이터에는 있지만 메뉴 마스터에는 없는 메뉴
unknown_menu_codes = ingredient_menu_codes - menu_codes

print("\n=== 식재료 데이터가 없는 메뉴 ===")
print("개수 :", len(missing_ingredient_codes))
print(list(missing_ingredient_codes)[:20])

print("\n=== 메뉴 마스터에 존재하지 않는 식재료 메뉴 코드 ===")
print("개수 :", len(unknown_menu_codes))
print(list(unknown_menu_codes)[:20])

# 메뉴별 식재료 개수 확인
ingredient_count = (
    ingredient_df
    .groupby("menu_fd_Code")
    .size()
    .sort_values(ascending=False)
)

print("\n=== 메뉴별 식재료 개수 통계 ===")
display(ingredient_count.describe())

print("\n=== 식재료가 가장 많은 메뉴 Top 10 ===")
display(ingredient_count.head(10))

메뉴 마스터 고유 코드 수 : 3250
식재료 데이터 고유 코드 수 : 3250

=== 식재료 데이터가 없는 메뉴 ===
개수 : 0
[]

=== 메뉴 마스터에 존재하지 않는 식재료 메뉴 코드 ===
개수 : 0
[]

=== 메뉴별 식재료 개수 통계 ===


count    3250.000000
mean        6.948308
std         4.603561
min         1.000000
25%         2.000000
50%         7.000000
75%        10.000000
max        27.000000
dtype: float64


=== 식재료가 가장 많은 메뉴 Top 10 ===


menu_fd_Code
D054013    27
D064024    25
D082061    24
D064030    22
D053126    22
D031066    21
D105020    21
D112032    21
D016021    21
D014037    20
dtype: int64

## 5. 국가표준식품성분 데이터 구조 확인

영양성분 엑셀 파일의 시트 목록과 각 시트의 크기를 확인하여
실제 데이터가 존재하는 시트를 식별한다.

In [8]:
nutrition_path = DATA_DIR / "food_nutrition.xlsx"

# 엑셀 파일의 시트 목록 확인
xls = pd.ExcelFile(nutrition_path)

print("=== 시트 목록 ===")
print(xls.sheet_names)

print("\n=== 시트별 데이터 크기 ===")

for sheet in xls.sheet_names:
    temp_df = pd.read_excel(
        nutrition_path,
        sheet_name=sheet
    )
    
    print(f"{sheet} : {temp_df.shape}")

=== 시트 목록 ===
['DB 설명', '국가표준식품성분 Database 10.0', '국가표준식품성분 Database 10.1', '국가표준식품성분 Database 10.2', '국가표준식품성분 Database 10.3', '국가표준식품성분 Database 10.4', 'DB 10.4 신규,교체,삭제 식품목록', '부록1)식품코드 연계표', '부록2)식품코드,국문명,영문명,학명 정보 ', '부록3)영양성분표기및단위']

=== 시트별 데이터 크기 ===
DB 설명 : (0, 0)
국가표준식품성분 Database 10.0 : (3272, 137)
국가표준식품성분 Database 10.1 : (3261, 137)
국가표준식품성분 Database 10.2 : (3312, 137)
국가표준식품성분 Database 10.3 : (3332, 137)
국가표준식품성분 Database 10.4 : (3368, 137)
DB 10.4 신규,교체,삭제 식품목록 : (188, 3)
부록1)식품코드 연계표 : (3369, 15)
부록2)식품코드,국문명,영문명,학명 정보  : (3366, 5)
부록3)영양성분표기및단위 : (135, 5)


## 6. 국가표준식품성분 Database 10.4 로드

국가표준식품성분 데이터 중 최신 버전인 10.4를 기준 데이터로 사용한다.
이후 MenuGen 식재료 데이터와 연결하기 위해 식품코드, 식품명 및 주요 영양성분 컬럼을 확인한다.

In [11]:
nutrition_df = pd.read_excel(
    nutrition_path,
    sheet_name="국가표준식품성분 Database 10.4"
)

print("nutrition_df shape :", nutrition_df.shape)

print("\n=== 컬럼 목록 ===")
for i, col in enumerate(nutrition_df.columns):
    print(i, col)

print("\n=== 상위 5행 ===")
display(nutrition_df.head())

print("\n=== 결측치 개수 상위 30개 ===")
display(
    nutrition_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .head(30)
)

nutrition_df shape : (3368, 137)

=== 컬럼 목록 ===
0 Unnamed: 0
1 *음영표시: 책자 수록 항목
2 Unnamed: 2
3 가식부 100g 당 (per 100g Edible Portion)
4 Unnamed: 4
5 일반성분 Proximates
6 Unnamed: 6
7 Unnamed: 7
8 Unnamed: 8
9 Unnamed: 9
10 Unnamed: 10
11 Unnamed: 11
12 Unnamed: 12
13 Unnamed: 13
14 Unnamed: 14
15 Unnamed: 15
16 Unnamed: 16
17 Unnamed: 17
18 Unnamed: 18
19 Unnamed: 19
20 Unnamed: 20
21 무기질 Minerals
22 Unnamed: 22
23 Unnamed: 23
24 Unnamed: 24
25 Unnamed: 25
26 Unnamed: 26
27 Unnamed: 27
28 Unnamed: 28
29 Unnamed: 29
30 Unnamed: 30
31 Unnamed: 31
32 Unnamed: 32
33 비타민 Vitamins
34 Unnamed: 34
35 Unnamed: 35
36 Unnamed: 36
37 Unnamed: 37
38 Unnamed: 38
39 Unnamed: 39
40 Unnamed: 40
41 Unnamed: 41
42 Unnamed: 42
43 Unnamed: 43
44 Unnamed: 44
45 Unnamed: 45
46 Unnamed: 46
47 Unnamed: 47
48 Unnamed: 48
49 Unnamed: 49
50 Unnamed: 50
51 Unnamed: 51
52 Unnamed: 52
53 Unnamed: 53
54 Unnamed: 54
55 Unnamed: 55
56 Unnamed: 56
57 Unnamed: 57
58 Unnamed: 58
59 Unnamed: 59
60 Unnamed: 60
61 Unnamed: 61
62 U

,Unnamed: 0,*음영표시: 책자 수록 항목,Unnamed: 2,가식부 100g 당 (per 100g Edible Portion),Unnamed: 4,일반성분 Proximates,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,식염상당량,폐기율
0,DB10.4\n색인,10개정 \n책자\n색인,식품군,식품명,출처,에너지,수분,단백질,지방,회분,...,도코사\n펜타에노산\n(22:5(n-3)),도코사\n헥사에노산\n(22:6(n-3)),오메가3 \n지방산,오메가6 \n지방산,총 트랜스\n지방산,트랜스 \n올레산(18:1(n-9)t),트랜스 \n리놀레산(18:2t),트랜스 \n리놀렌산(18:3t),NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,kcal,g,g,g,g,...,mg,mg,g,g,g,mg,mg,mg,g,%
2,1,1,곡류 및 그 제품,"귀리, 겉귀리, 도정, 생것",농진청('21),388,11.7,9.88,8.84,1.56,...,0,0,0.08,2.77,0.02,2.75,10.91,4.5,0,0
3,3381,NaN,곡류 및 그 제품,"귀리, 겉귀리, 도정, 밥",농진청('21),197,55.5,5.01,4.7,0.72,...,0,0,0.04,1.4,0.01,1.84,9.06,3.35,0,0
4,2,2,곡류 및 그 제품,"귀리, 쌀귀리, 도정, 생것",농진청('20),388,11.6,11.14,8.9,1.7,...,0,0,0.07,2.8,0.01,2.97,3.58,5.25,0,0



=== 결측치 개수 상위 30개 ===


*음영표시: 책자 수록 항목                         2152
Unnamed: 0                                 1
Unnamed: 2                                 1
가식부 100g 당 (per 100g Edible Portion)       1
Unnamed: 4                                 1
폐기율                                        1
콜레스테롤                                      1
식염상당량                                      1
일반성분 Proximates                            0
Unnamed: 8                                 0
Unnamed: 7                                 0
Unnamed: 6                                 0
Unnamed: 9                                 0
Unnamed: 13                                0
Unnamed: 10                                0
Unnamed: 11                                0
Unnamed: 12                                0
Unnamed: 17                                0
Unnamed: 18                                0
Unnamed: 19                                0
Unnamed: 20                                0
무기질 Minerals                               0
Unnamed: 1

## 7. 국가표준식품성분 DB 10.4 헤더 정제

엑셀의 실제 컬럼명은 두 번째 행에 존재하며,
그 아래 단위 행을 제거하여 실제 식품 데이터만 구성한다.

In [32]:
# DB10.4 원본 로드
nutrition_raw = pd.read_excel(
    nutrition_path,
    sheet_name="국가표준식품성분 Database 10.4",
    header=None
)

# 헤더 구성 정보
upper_header = nutrition_raw.iloc[0]
detail_header = nutrition_raw.iloc[1]
unit_row = nutrition_raw.iloc[2].copy()

# 세부 컬럼명이 있으면 사용하고,
# 비어 있으면 상위분류명을 컬럼명으로 사용
column_names = detail_header.copy()

column_names = column_names.where(
    column_names.notna(),
    upper_header
)


# 컬럼명 정규화 함수
def clean_column_names(columns):
    return (
        pd.Index(columns)
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


column_names = clean_column_names(column_names)

# 실제 식품 데이터
nutrition_df = (
    nutrition_raw
    .iloc[3:]
    .reset_index(drop=True)
)

nutrition_df.columns = column_names

# 단위 정보의 인덱스도 동일한 컬럼명으로 연결
unit_row.index = column_names


print("nutrition_df shape :", nutrition_df.shape)
display(nutrition_df.head(2))

nutrition_df shape : (3366, 137)


1,DB10.4 색인,10개정 책자 색인,식품군,식품명,출처,에너지,수분,단백질,지방,회분,...,도코사 펜타에노산 (22:5(n-3)),도코사 헥사에노산 (22:6(n-3)),오메가3 지방산,오메가6 지방산,총 트랜스 지방산,트랜스 올레산(18:1(n-9)t),트랜스 리놀레산(18:2t),트랜스 리놀렌산(18:3t),식염상당량,폐기율
0,1,1,곡류 및 그 제품,"귀리, 겉귀리, 도정, 생것",농진청('21),388,11.7,9.88,8.84,1.56,...,0,0,0.08,2.77,0.02,2.75,10.91,4.5,0,0
1,3381,NaN,곡류 및 그 제품,"귀리, 겉귀리, 도정, 밥",농진청('21),197,55.5,5.01,4.7,0.72,...,0,0,0.04,1.4,0.01,1.84,9.06,3.35,0,0


## 8. DB10.4 영양성분 표기 규칙

국가표준식품성분 DB10.4 설명문의 표기 기준을 적용한다.

- `-` : 측정되지 않은(unmeasured) 결측값
- `Tr` : 검출되었으나 정량 가능한 최소 농도 이하의 미량(trace)
- `(수치)` : 인용되었거나 재료량을 이용해 환산한 수치
- `(Tr)` : 괄호 표기가 적용된 미량값

원본값은 보존하고 계산용 수치와 데이터 상태를 별도로 관리한다.

In [33]:
def parse_nutrition_value(value):
    if pd.isna(value):
        return np.nan, "missing"

    value = str(value).strip()

    if value == "-":
        return np.nan, "unmeasured"

    if value.lower() == "tr":
        return np.nan, "trace"

    if value.lower() == "(tr)":
        return np.nan, "quoted_or_converted_trace"

    if re.fullmatch(r"\([\d,]+(?:\.\d+)?\)", value):
        numeric_value = value[1:-1].replace(",", "")
        return float(numeric_value), "quoted_or_converted"

    numeric_value = pd.to_numeric(
        value.replace(",", ""),
        errors="coerce"
    )

    if pd.notna(numeric_value):
        return float(numeric_value), "numeric"

    return np.nan, "unknown"

## 9. 영양성분 컬럼 그룹 구성

DB10.4의 상위 헤더 정보를 이용하여 전체 컬럼을 영양성분 그룹별로 분류한다.

전체 컬럼을 개별적으로 확인하지 않고,
영양성분 그룹 단위로 활용 범위를 먼저 선정한 뒤
필요한 세부 컬럼만 추출한다.

In [38]:
# DB10.4 상위 헤더를 각 컬럼에 전달
group_names = nutrition_raw.iloc[0].ffill()

# 상위 헤더명 정규화
group_names = clean_column_names(group_names)

# 컬럼 메타데이터 구성
nutrition_meta_df = pd.DataFrame({
    "column": nutrition_df.columns,
    "group": group_names,
    "unit": unit_row.values
})

# 식품 기본정보는 별도 그룹으로 지정
info_cols = [
    "DB10.4 색인",
    "10개정 책자 색인",
    "식품군",
    "식품명",
    "출처"
]

nutrition_meta_df.loc[
    nutrition_meta_df["column"].isin(info_cols),
    "group"
] = "식품정보"


# 단위가 있는 컬럼 = 수치형 전처리 대상
# 영양성분뿐 아니라 식염상당량, 폐기율도 포함
value_cols = (
    nutrition_meta_df
    .loc[nutrition_meta_df["unit"].notna(), "column"]
    .tolist()
)


print("식품 정보 컬럼 수 :", len(info_cols))
print("수치형 전처리 대상 :", len(value_cols))


# 그룹별 구조 확인
group_summary_df = (
    nutrition_meta_df
    .groupby("group", dropna=False)
    .agg(
        컬럼수=("column", "count"),
        컬럼목록=("column", list)
    )
    .reset_index()
)

display(group_summary_df)

식품 정보 컬럼 수 : 5
수치형 전처리 대상 : 132


,group,컬럼수,컬럼목록
0,무기질 Minerals,12,"[칼슘, 철, 마그네슘, 인, 칼륨, 나트륨, 아연, 구리, 망간, 셀레늄, 몰리브..."
1,비타민 Vitamins,33,"[비타민 A, 레티놀, 베타카로틴, 티아민, 리보플라빈, 니아신, 니아신당량(NE)..."
2,식염상당량,1,[식염상당량]
3,식품정보,5,"[DB10.4 색인, 10개정 책자 색인, 식품군, 식품명, 출처]"
4,아미노산 Amino acids,21,"[총 아미노산, 총 필수 아미노산, 이소류신, 류신, 라이신, 메티오닌, 페닐알라닌..."
5,일반성분 Proximates,16,"[에너지, 수분, 단백질, 지방, 회분, 탄수화물, 당류, 자당, 포도당, 과당, ..."
6,지방산 Fatty acids,47,"[총 지방산, 총 필수 지방산, 총 포화 지방산, 부티르산 (4:0), 카프로산 (..."
7,콜레스테롤,1,[콜레스테롤]
8,폐기율,1,[폐기율]


## 10. 수치형 컬럼 전체 정제

In [39]:
# 원본 보존
nutrition_clean_df = nutrition_df.copy()

# 원본 값의 상태정보 저장
nutrition_status_df = pd.DataFrame(
    index=nutrition_df.index
)


for col in value_cols:
    parsed = nutrition_df[col].map(parse_nutrition_value)

    # 계산에 사용할 숫자값
    nutrition_clean_df[col] = parsed.map(
        lambda x: x[0]
    )

    # 값의 원래 상태
    nutrition_status_df[col] = parsed.map(
        lambda x: x[1]
    )


print("전처리 대상 컬럼 :", len(value_cols))
print("정제 데이터 shape :", nutrition_clean_df.shape)


print("\n=== 값 상태 집계 ===")

status_counts = (
    nutrition_status_df
    .stack()
    .value_counts()
)

display(status_counts)

전처리 대상 컬럼 : 132
정제 데이터 shape : (3366, 137)

=== 값 상태 집계 ===


numeric                      305480
unmeasured                   133405
quoted_or_converted            4832
trace                           431
quoted_or_converted_trace       164
Name: count, dtype: int64

## 결측치 

In [40]:
missing_report = pd.DataFrame({
    "column": value_cols,
    "missing_count": [
        nutrition_clean_df[col].isna().sum()
        for col in value_cols
    ]
})

missing_report["missing_rate"] = (
    missing_report["missing_count"]
    / len(nutrition_clean_df)
    * 100
).round(2)

missing_report = missing_report.sort_values(
    "missing_rate",
    ascending=False
).reset_index(drop=True)

display(missing_report)

,column,missing_count,missing_rate
0,비타민 B6,1777,52.79
1,타우린,1668,49.55
2,비타민 K2,1607,47.74
3,니코틴산,1502,44.62
4,니코틴아미드,1502,44.62
...,...,...,...
127,지방,18,0.53
128,회분,12,0.36
129,수분,9,0.27
130,단백질,1,0.03


## 중복 검증

In [41]:
print("전체 식품 수 :", len(nutrition_clean_df))

print(
    "DB10.4 색인 중복 :",
    nutrition_clean_df["DB10.4 색인"].duplicated().sum()
)

print(
    "완전 중복 행 :",
    nutrition_clean_df.duplicated().sum()
)

print(
    "식품명 중복 :",
    nutrition_clean_df["식품명"].duplicated().sum()
)

전체 식품 수 : 3366
DB10.4 색인 중복 : 0
완전 중복 행 : 0
식품명 중복 : 0


## 이상치

In [43]:
outlier_rows = []

for col in value_cols:
    series = nutrition_clean_df[col].dropna()

    if series.empty:
        continue

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3 - q1

    lower = q1 - (1.5 * iqr)
    upper = q3 + (1.5 * iqr)

    outlier_count = (
        (series < lower) |
        (series > upper)
    ).sum()

    outlier_rows.append({
        "column": col,
        "min": series.min(),
        "max": series.max(),
        "Q1": q1,
        "Q3": q3,
        "IQR_outlier_count": outlier_count
    })


outlier_report = pd.DataFrame(outlier_rows)

display(
    outlier_report.sort_values(
        "IQR_outlier_count",
        ascending=False
    )
)

,column,min,max,Q1,Q3,IQR_outlier_count
131,폐기율,0.0,91.00,0.00,5.0000,692
122,도코사 펜타에노산 (22:5(n-3)),0.0,2991.00,0.00,0.4175,566
104,미리스톨레산 (14:1),0.0,687.67,0.00,0.0000,539
30,베타카로틴,0.0,47375.00,0.00,98.7500,537
110,에루크산 (22:1),0.0,6231.61,0.00,0.3100,513
...,...,...,...,...,...,...
43,엽산_ 합성 엽산,0.0,523.00,0.00,0.0000,34
62,총 필수 아미노산,0.0,40550.00,437.50,6611.0000,34
64,류신,0.0,7300.00,69.25,1135.0000,31
70,발린,0.0,5400.00,52.00,723.5000,31
